# MetaAuditor Adversity - Unsloth Training Script
This notebook fine-tunes Llama-3-8B-Instruct to become a flawless Enterprise Auditor Agent using the `dataset.jsonl` synthetic trajectories.

**Prerequisites:**
1. Open this file in Google Colab.
2. Go to `Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU`.
3. Upload `dataset.jsonl` to the Colab files section (left menu).

🚨 **CRITICAL FIX FOR MODULE ERRORS:** 🚨
After running the `pip install` cell below, Google Colab often fails to load newly installed packages immediately. You **MUST** restart the session (`Runtime -> Restart session` or `Ctrl+M .`) BEFORE moving to Step 1, otherwise you will encounter a `ModuleNotFoundError: No module named 'unsloth'`.

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.28" "trl==0.8.6" peft accelerate bitsandbytes

## 1. Load the Pre-Trained Model
Using Unsloth, we load the model in 4-bit precision to save GPU memory.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Attach LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

## 2. Load the MetaAuditor Dataset
We format our trajectories to HuggingFace Dataset format. If you forgot to upload `dataset.jsonl`, the cell below will ask you to upload it.

In [ ]:
import os
from google.colab import files

if not os.path.exists('dataset.jsonl'):
    print('dataset.jsonl not found. Please upload it now:')
    uploaded = files.upload()
    if 'dataset.jsonl' not in uploaded:
        raise FileNotFoundError('You must upload dataset.jsonl to continue.')

from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3",
    mapping = {"role": "from", "content": "value", "user": "user", "assistant": "assistant"}
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts }

dataset = load_dataset("json", data_files="dataset.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)

## 3. Train the Model
Fine-tuning begins here. On a T4 GPU, this should take 15-30 minutes for 1,500 samples.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

## 4. Push to HuggingFace
Upload the fine-tuned LoRA adapters to your HuggingFace account so it can be evaluated by OpenEnv.

In [ ]:
# Replace with your actual HuggingFace Write Token
HF_TOKEN = "hf_YOUR_TOKEN_HERE"

model.push_to_hub("Dhusyanth03/meta-auditor-agent-lora", token=HF_TOKEN)
tokenizer.push_to_hub("Dhusyanth03/meta-auditor-agent-lora", token=HF_TOKEN)